In [1]:
import argparse
import pandas as pd
import numpy as np
from preprocess import preprocess
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_recall_curve, precision_score, recall_score, average_precision_score, roc_curve, auc, confusion_matrix, mean_squared_error,classification_report
import time

import matplotlib.pyplot as plt
from keras.utils import to_categorical

trainset = pd.read_csv('./data/NSL-KDD/KDDTrain+.txt', sep=",", header=None)
testset = pd.read_csv('./data/NSL-KDD/KDDTest+.txt', sep=",", header=None)


processor = preprocess()
print("数据预处理....")
df_train, df_test, train_Normal, train_R2L, train_U2R, train_Dos, train_Probe,test_Normal, test_R2L, test_U2R, test_Dos, test_Probe,train_Attack,test_Attack = processor.create_df(df_train=trainset, df_test=testset)
# normal_df, R2L_df, R2L_df_train, R2L_df_test = processor.create_df(df_train=trainset, df_test=testset)
print("已完成数据预处理")


Using TensorFlow backend.
D:\Anaconda3\envs\TF_36aa\lib\site-packages\tensorflow\python\framework\dtypes.py:523: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
D:\Anaconda3\envs\TF_36aa\lib\site-packages\tensorflow\python\framework\dtypes.py:524: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
D:\Anaconda3\envs\TF_36aa\lib\site-packages\tensorflow\python\framework\dtypes.py:525: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
D:\Anaconda3\envs\TF_36aa\lib\site-packages\tensorflow\python\framework\dtype

数据预处理....
已完成数据预处理


In [2]:
dt={'Dos':pd.Series([len(train_Dos),len(test_Dos)],index=['Train','Test']),
   'Probe':pd.Series([len(train_Probe),len(test_Probe)],index=['Train','Test']),
   'R2L':pd.Series([len(train_R2L),len(test_R2L)],index=['Train','Test']),
   'U2R':pd.Series([len(train_U2R),len(test_U2R)],index=['Train','Test']),
   'Normal':pd.Series([len(train_Normal),len(test_Normal)],index=['Train','Test']),
   'Total_attack':pd.Series([len(train_Attack),len(test_Attack)],index=['Train','Test']),
   'Total':pd.Series([len(df_train),len(df_test)],index=['Train','Test'])}
type_df=pd.DataFrame(dt)
cols = ['Dos','Probe','R2L','U2R','Normal','Total_attack','Total']
type_df = type_df[cols]
display(type_df)


,Dos,Probe,R2L,U2R,Normal,Total_attack,Total
Train,11656,45927,995,52,67343,58630,125973
Test,2421,7460,2885,67,9711,12833,22544


In [3]:
#随机划分的全局二分类
from sklearn.model_selection import train_test_split

train_Bi = df_train
test_Bi = df_test

combined_data = pd.concat([train_Bi, test_Bi])
combined_data = combined_data.replace(4,1).replace(3,1).replace(2,1)
data_x = combined_data.drop(['attack_type'], axis=1) # droped label
data_y = combined_data.loc[:,['attack_type']]
# del combined_data # free mem
X_train, X_test, y_train, y_test = train_test_split(data_x, data_y, test_size=.20, random_state=42) # TODO


In [4]:
imbalance_class_data = pd.concat([train_Normal, train_R2L, train_U2R,test_Normal, test_R2L, test_U2R])
imbalance_class_data = imbalance_class_data.replace(4,1).replace(3,1).replace(2,1)

In [8]:
combined_data.head()

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_srv_count,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate,attack_type
0,0.0,0.5,0.289855,0.9,3.558064e-07,0.000000e+00,0,0.0,0.0,0.0,...,0.098039,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,0
1,0.0,1.0,0.637681,0.9,1.057999e-07,0.000000e+00,0,0.0,0.0,0.0,...,0.003922,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,0
2,0.0,0.5,0.710145,0.5,0.000000e+00,0.000000e+00,0,0.0,0.0,0.0,...,0.101961,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,1
3,0.0,0.5,0.347826,0.9,1.681203e-07,6.223962e-06,0,0.0,0.0,0.0,...,1.000000,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,0
4,0.0,0.5,0.347826,0.9,1.442067e-07,3.206260e-07,0,0.0,0.0,0.0,...,1.000000,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0


In [5]:
imbalance_class_data.to_csv("./data/imbalance_class_data.csv",mode='w+',sep=',', index=False, header=True)
# combined_data.to_csv("./data/df_train.csv",mode='w+',sep=',', index=False, header=True)
# data_x.to_csv("./data/data_x.csv",mode='w+',sep=',', index=False, header=None)
# data_y.to_csv("./data/data_y.csv",mode='w+',sep=',', index=False, header=None)

In [4]:
#非随机划分全局二分类

X_train_Bi,y_train_Bi = processor.split_df(df_train)
y_train_Bi = y_train_Bi.replace(4,1).replace(3,1).replace(2,1)

X_test_Bi,y_test_Bi = processor.split_df(df_test)
y_test_Bi = y_test_Bi.replace(4,1).replace(3,1).replace(2,1)


In [5]:
# X = X_train_Bi
# Y = y_train_Bi
# C = y_test_Bi
# T = X_test_Bi

X = X_train
T = X_test
Y = y_train
C = y_test

In [6]:
X.head(3)

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_count,Dst_host_srv_count,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate
21934,0.0,0.5,0.782609,0.9,9.036468e-07,2.534472e-07,0,0.0,0.0,0.0,...,0.160784,0.650980,0.63,0.15,0.02,0.02,0.0,0.0,0.0,0.0
2030,0.0,0.5,0.347826,0.9,5.666476e-06,3.150988e-03,0,0.0,0.0,0.0,...,0.109804,1.000000,1.00,0.00,0.04,0.02,0.0,0.0,0.0,0.0
10407,0.0,0.5,0.782609,0.9,6.674088e-07,2.519204e-07,0,0.0,0.0,0.0,...,0.815686,0.960784,0.96,0.01,0.00,0.01,0.0,0.0,0.0,0.0


In [7]:
C.head(3)

,attack_type
38166,0
122124,0
11263,0


In [8]:
T.head(3)

,Duration,Protocol_type,Service,Flag,Src_bytes,Dst_bytes,Land,Wrong_fragment,Urgent,Hot,...,Dst_host_count,Dst_host_srv_count,Dst_host_same_srv_rate,Dst_host_diff_srv_rate,Dst_host_same_src_port_rate,Dst_host_srv_diff_host_rate,Dst_host_serror_rate,Dst_host_srv_serror_rate,Dst_host_rerror_rate,Dst_host_srv_rerror_rate
38166,0.0,1.0,0.173913,0.9,3.188489e-08,8.779045e-08,0,0.0,0.0,0.0,...,1.000000,0.972549,0.97,0.01,0.00,0.00,0.0,0.0,0.0,0.0
122124,0.0,0.5,0.347826,0.9,1.797148e-07,9.718022e-07,0,0.0,0.0,0.0,...,0.113725,1.000000,1.00,0.00,0.03,0.03,0.0,0.0,0.0,0.0
11263,0.0,0.5,0.347826,0.9,4.934291e-06,1.454016e-03,0,0.0,0.0,0.0,...,0.043137,0.945098,1.00,0.00,0.09,0.08,0.0,0.0,0.0,0.0


In [9]:
import numpy as np
import pandas as pd
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn import svm
from sklearn.metrics import classification_report
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import Normalizer
from pandas_ml import ConfusionMatrix
# traindata = pd.read_csv('UNSW_NB15_training-set.csv', header=None)
# testdata = pd.read_csv('UNSW_NB15_testing-set.csv', header=None)
# traindata = pd.read_csv('kddtrain.csv', header=None)
# testdata = pd.read_csv('kddtest.csv', header=None)


# X = traindata.iloc[:,1:42]
# Y = traindata.iloc[:,0]
# C = testdata.iloc[:,0]
# T = testdata.iloc[:,1:42]


scaler = Normalizer().fit(X)
trainX = scaler.transform(X)

scaler = Normalizer().fit(T)
testT = scaler.transform(T)


traindata = np.array(trainX)
trainlabel = np.array(Y)

testdata = np.array(testT)
testlabel = np.array(C)
testlabel = testlabel.flatten()

model = LogisticRegression()
model.fit(traindata, trainlabel)


# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(29704,)
(29704,)
***************************************************************


In [10]:
model = LogisticRegression()
model.fit(traindata, trainlabel)
print(model)

# make predictions
expected = testlabel
predicted = model.predict(testdata)

#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)


cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")



D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\linear_model\logistic.py:432: FutureWarning: Default solver will be changed to 'lbfgs' in 0.22. Specify a solver to silence this warning.
  FutureWarning)
D:\Anaconda3\envs\TF_36a\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='warn',
          n_jobs=None, penalty='l2', random_state=None, solver='warn',
          tol=0.0001, verbose=0, warm_start=False)
(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 13825
NegativeTest: 15879
TP: 13027
TN: 14652
FP: 798
FN: 1227
TPR: 0.913918899958
TNR: 0.948349514563
PPV: 0.942278481013
NPV: 0.922728131494
FPR: 0.0516504854369
FDR: 0.0577215189873
FNR: 0.0860811000421
ACC: 0.931827363318
F1_score: 0.927882047081
MCC: 0.863636428317
informedness: 0.862268414521
markedness: 0.865006612507
prevalence: 0.479868031242
LRP: 17.6942944917
LRN: 0.0907693827225
DOR: 194.936816369
FOR: 0.0772718685056
Predicted  False   True  __all__
Actual                          
False      14652    798    15450
True        1227  13027    14254
__all__    15879  13825    29704
(29704,)
(29704,)
***************************************************

In [12]:
# fit a Naive Bayes model to the data
model = GaussianNB()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)

# expected = expected.flatten()
#predicted = predicted.reshape(len(predicted),1)
print(expected.shape)
print(predicted.shape)

print(type(expected))

cm = ConfusionMatrix(expected, predicted)

expected = np.array(expected)
predicted = np.array(predicted)



cm.print_stats()

np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')

print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")





# fit a k-nearest neighbor model to the data
model = KNeighborsClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()


# cm = metrics.confusion_matrix(expected, predicted)
# print(cm)
# tpr = float(cm[0][0])/np.sum(cm[0])
# fpr = float(cm[1][1])/np.sum(cm[1])
# print("%.3f" %tpr)
# print("%.3f" %fpr)
# print("Accuracy")
# print("%.3f" %ACC)
# print("precision")
# print("%.3f" %precision)
# print("recall")
# print("%.3f" %recall)
# print("f-score")
# print("%.3f" %f1)
# print("fpr")
# print("%.3f" %fpr)
# print("tpr")
# print("%.3f" %tpr)
print("***************************************************************")



model = DecisionTreeClassifier()
model.fit(traindata, trainlabel)
print(model)
# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()
print("***************************************************************")

print("AdaBoostClassifier")
model = AdaBoostClassifier(n_estimators=100)
model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model

#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



print("RandomForestClassifier")
model = RandomForestClassifier(n_estimators=100)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
#expected = expected.flatten()

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("end****end***********************************************************")




D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


GaussianNB(priors=None, var_smoothing=1e-09)
(29704,)
(29704,)
<class 'numpy.ndarray'>
population: 29704
P: 14254
N: 15450
PositiveTest: 14211
NegativeTest: 15493
TP: 12455
TN: 13694
FP: 1756
FN: 1799
TPR: 0.873789813386
TNR: 0.886343042071
PPV: 0.876433748505
NPV: 0.883883043955
FPR: 0.113656957929
FDR: 0.123566251495
FNR: 0.126210186614
ACC: 0.880319148936
F1_score: 0.875109783945
MCC: 0.760224818396
informedness: 0.760132855457
markedness: 0.76031679246
prevalence: 0.479868031242
LRP: 7.68795707108
LRN: 0.142394288242
DOR: 53.9906281774
FOR: 0.116116956045
Predicted  False   True  __all__
Actual                          
False      13694   1756    15450
True        1799  12455    14254
__all__    15493  14211    29704
(29704,)
(29704,)
***************************************************************


D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:40: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().


KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
           metric_params=None, n_jobs=None, n_neighbors=5, p=2,
           weights='uniform')
(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 14260
NegativeTest: 15444
TP: 14098
TN: 15288
FP: 162
FN: 156
TPR: 0.989055703662
TNR: 0.989514563107
PPV: 0.988639551192
NPV: 0.989898989899
FPR: 0.0104854368932
FDR: 0.0113604488079
FNR: 0.0109442963379
ACC: 0.989294371128
F1_score: 0.988847583643
MCC: 0.978554403801
informedness: 0.978570266769
markedness: 0.978538541091
prevalence: 0.479868031242
LRP: 94.3266087752
LRN: 0.0110602680809
DOR: 8528.41975309
FOR: 0.010101010101
Predicted  False   True  __all__
Actual                          
False      15288    162    15450
True         156  14098    14254
__all__    15444  14260    29704
(29704,)
(29704,)
***************************************************************
DecisionTreeClassifier(class_weight=None, criterion='gini', max_depth=None,
          

D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


population: 29704
P: 14254
N: 15450
PositiveTest: 14117
NegativeTest: 15587
TP: 13717
TN: 15050
FP: 400
FN: 537
TPR: 0.962326364529
TNR: 0.974110032362
PPV: 0.971665367996
NPV: 0.965548213255
FPR: 0.0258899676375
FDR: 0.028334632004
FNR: 0.0376736354707
ACC: 0.968455426879
F1_score: 0.966973317825
MCC: 0.936824908478
informedness: 0.936436396892
markedness: 0.937213581251
prevalence: 0.479868031242
LRP: 37.1698558299
LRN: 0.0386749281078
DOR: 961.084031657
FOR: 0.0344517867454
Predicted  False   True  __all__
Actual                          
False      15050    400    15450
True         537  13717    14254
__all__    15587  14117    29704
(29704,)
(29704,)
***************************************************************
RandomForestClassifier


D:\Anaconda3\envs\TF_36m\lib\site-packages\ipykernel_launcher.py:135: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().


population: 29704
P: 14254
N: 15450
PositiveTest: 14219
NegativeTest: 15485
TP: 14141
TN: 15372
FP: 78
FN: 113
TPR: 0.99207240073
TNR: 0.994951456311
PPV: 0.994514382165
NPV: 0.992702615434
FPR: 0.00504854368932
FDR: 0.00548561783529
FNR: 0.00792759927038
ACC: 0.993569889577
F1_score: 0.993291890563
MCC: 0.987120422596
informedness: 0.98702385704
markedness: 0.987216997599
prevalence: 0.479868031242
LRP: 196.506648606
LRN: 0.00796782518393
DOR: 24662.5200817
FOR: 0.00729738456571
Predicted  False   True  __all__
Actual                          
False      15372     78    15450
True         113  14141    14254
__all__    15485  14219    29704
(29704,)
(29704,)
end****end***********************************************************


In [16]:
#model = svm.OneClassSVM( kernel='linear', random_state=0)
# model = svm.LinearSVC(C=20)
model = svm.SVC(kernel='linear')#调参
#model = svm.SVC(kernel='linear', C=1.0, random_state=0)
model = model.fit(traindata, trainlabel)

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
print(expected.shape)
print(predicted.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")



D:\Anaconda3\envs\TF_36m\lib\site-packages\sklearn\utils\validation.py:752: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 13542
NegativeTest: 16162
TP: 12900
TN: 14808
FP: 642
FN: 1354
TPR: 0.905009120247
TNR: 0.958446601942
PPV: 0.952591936198
NPV: 0.916223239698
FPR: 0.0415533980583
FDR: 0.0474080638015
FNR: 0.0949908797531
ACC: 0.932803662806
F1_score: 0.928191106634
MCC: 0.866131303644
informedness: 0.863455722189
markedness: 0.868815175897
prevalence: 0.479868031242
LRP: 21.7794250901
LRN: 0.0991092039563
DOR: 219.751791162
FOR: 0.0837767603019
Predicted  False   True  __all__
Actual                          
False      14808    642    15450
True        1354  12900    14254
__all__    16162  13542    29704
(29704,)
(29704,)
***************************************************************


In [17]:
print(f"Classification report for classifier {model}:\n"
      f"{metrics.classification_report(expected, predicted)}\n")

Classification report for classifier SVC(C=1.0, cache_size=200, class_weight=None, coef0=0.0,
  decision_function_shape='ovr', degree=3, gamma='auto_deprecated',
  kernel='linear', max_iter=-1, probability=False, random_state=None,
  shrinking=True, tol=0.001, verbose=False):
              precision    recall  f1-score   support

           0       0.92      0.96      0.94     15450
           1       0.95      0.91      0.93     14254

   micro avg       0.93      0.93      0.93     29704
   macro avg       0.93      0.93      0.93     29704
weighted avg       0.93      0.93      0.93     29704




In [10]:
from keras.models import Sequential, Model
from keras.layers import Dense, Dropout, Activation, Embedding
from keras.wrappers.scikit_learn import KerasClassifier
import h5py
from keras import callbacks
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from keras.utils import to_categorical
def build_model():
    # 1. define the network
    model = Sequential()
    #model = Model()
    model.add(Dense(1024,input_dim=41,activation='relu'))  
    model.add(Dropout(0.01))
    model.add(Dense(1))
    model.add(Activation('sigmoid'))
    # try using different optimizers and different optimizer configs
    model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])   
   # model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])   
    return model

In [11]:
#DNN
checkpointer = callbacks.ModelCheckpoint(filepath="./DNNResult/checkpoint-{epoch:02d}.hdf5", verbose=1, save_best_only=True, monitor='loss')
#csv_logger = CSVLogger('./DNNResult/training_set_dnnanalysis.csv',separator=',', append=False)
model = KerasClassifier(build_fn=build_model, epochs=1, batch_size=64)
#model.fit(traindata, trainlabel, callbacks=[checkpointer,csv_logger])
model.fit(traindata, trainlabel, callbacks=[checkpointer])
#model.save("DNNResult/dnn1layer_model.hdf5")

# make predictions
expected = testlabel
predicted = model.predict(testdata)
# summarize the fit of the model
expected = expected.flatten()
predicted = predicted.flatten()
print(predicted.shape)
print(expected.shape)

cm = ConfusionMatrix(expected, predicted)
expected = np.array(expected)
predicted = np.array(predicted)
cm.print_stats()
np.savetxt('expected.txt', expected, fmt='%01d')
np.savetxt('predicted.txt',predicted , fmt='%01d')
print(cm)
print(expected.shape)
print(predicted.shape)
cm.stats()

print("***************************************************************")


Epoch 1/1
118813/118813 [==============================] - 7s 61us/step - loss: 0.1339 - acc: 0.9499

Epoch 00001: loss improved from inf to 0.13388, saving model to ./DNNResult/checkpoint-01.hdf5
(29704,)
(29704,)
population: 29704
P: 14254
N: 15450
PositiveTest: 14095
NegativeTest: 15609
TP: 13652
TN: 15007
FP: 443
FN: 602
TPR: 0.957766241055
TNR: 0.971326860841
PPV: 0.968570415041
NPV: 0.961432506887
FPR: 0.0286731391586
FDR: 0.0314295849592
FNR: 0.0422337589449
ACC: 0.964819552922
F1_score: 0.96313802956
MCC: 0.929547900598
informedness: 0.929093101897
markedness: 0.930002921928
prevalence: 0.479868031242
LRP: 33.402908407
LRN: 0.0434804808222
DOR: 768.227668494
FOR: 0.0385674931129
Predicted  False   True  __all__
Actual                          
False      15007    443    15450
True         602  13652    14254
__all__    15609  14095    29704
(29704,)
(29704,)
***************************************************************
